# Bloque 0 – Dependencias, rutas y parámetros fiscales

In [81]:
# libraries
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# -----------------------------------------------------
# PATH CONFIGURATION
# -----------------------------------------------------
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from config.paths import RAW_DIR, PROCESSED_DIR

# -----------------------------------------------------
# TAX BRACKETS BY YEAR
# -----------------------------------------------------
TAX_BRACKETS = {
    2018: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ],
    2024: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ]
}

# -----------------------------------------------------
# DEFLATORS (base 2018)
# -----------------------------------------------------
DEFLATORS = {
    2018: 1.0,
    2024: (100.0 / 133.543876114864)
}

# Bloque 1 – Clasificación de claves ENIGH por componente CEQ

In [82]:
# =====================================================
# INCOME COMPONENT MAPPINGS (by year)
# =====================================================

# Trabajo principal (job 1)
LABOR_KEYS_MAIN = [
    "P001", "P002", "P003", "P004", "P005", "P006", "P007",
    "P008", "P009", "P011", "P012", "P013"
]

# Trabajo secundario (job 2)
LABOR_KEYS_SECONDARY = [
    "P014", "P015", "P016", "P018", "P019", "P020", "P021", "P022"
]

# Otros laborales (autoempleo, menores) – siempre informales
LABOR_KEYS_OTHER = [
    "P067",  # menores de 12 años
    "P068", "P069", "P070", "P071", "P072", "P073", "P074", "P075",
    "P076", "P077", "P078", "P079", "P080", "P081"
]

# Private transfers keys
PRIVATE_TRANSFER_KEYS = ["P039", "P040", "P041"]

# Public direct transfers – 2018
DIRECT_TRANSFERS_2018 = [
    "P042",  # Prospera
    "P043",  # PROCAMPO
    "P044",  # Pensión Adultos Mayores
    "P045",  # Otros programas para adultos mayores
    "P046",  # Tarjeta SinHambre
    "P047",  # Empleo temporal
    "P048",  # Otros programas sociales
    "P038"   # Becas de gobierno (no Prospera)
]

# Public direct transfers – 2024
DIRECT_TRANSFERS_2024 = [
    "P101",  # Becas Benito Juárez (básica)
    "P102",  # Beca Universal Educación Media Superior
    "P103",  # Jóvenes Escribiendo el Futuro
    "P104",  # Pensión Adultos Mayores (65 y más)
    "P105",  # Pensión para personas con discapacidad
    "P106",  # Apoyo estancias infantiles / madres trabajadoras
    "P107",  # Seguro de Vida para Jefas de Familia
    "P108",  # Jóvenes Construyendo el Futuro
    "P043",  # PROCAMPO / Producción para el Bienestar
    "P045",  # Otros programas para adultos mayores
    "P038"   # Becas de gobierno (no Prospera)
]

# Excluded keys (not used in income definition)
EXCLUDED_KEYS = [
    "P049", "P051", "P052", "P053", "P054", "P055", "P056", "P057",
    "P058", "P059", "P060", "P061", "P062", "P063", "P064", "P065", "P066"
]

# Bloque 2 – Funciones de cálculo (gross‑up laboral y cálculo de ISR)

In [83]:
# =====================================================
# FUNCTIONS
# =====================================================

def gross_up_labor(net_annual, brackets):
    """
    Devuelve el ingreso bruto anual a partir del neto anual,
    invirtiendo la tarifa del ISR.
    Si net_annual es 0, retorna 0.
    """
    gross = pd.Series(np.nan, index=net_annual.index)
    net = net_annual.fillna(0)
    for b in brackets:
        gross_temp = (net + b["cf"] - b["tm"] * b["li"]) / (1 - b["tm"])
        mask = gross.isna() & (gross_temp >= b["li"]) & (gross_temp <= b["ls"])
        gross[mask] = gross_temp[mask]
    gross[net == 0] = 0.0
    gross = gross.fillna(0)
    return gross


def calculate_isr(income_annual, brackets):
    """
    Calcula el ISR anual sobre un ingreso bruto.
    """
    if income_annual <= 0:
        return 0.0
    for b in brackets:
        if b["li"] <= income_annual <= b["ls"]:
            return b["cf"] + b["tm"] * (income_annual - b["li"])
    return 0.0


def compute_gross_and_tax(net_annual, is_formal, brackets):
    """
    Retorna una tupla (gross_annual, isr_annual) para un ingreso laboral.
    - Si is_formal == 1: se aplica gross‑up y se calcula el ISR.
    - Si is_formal == 0: bruto = neto, ISR = 0.
    """
    net_annual = net_annual.fillna(0)
    gross = pd.Series(0.0, index=net_annual.index)
    isr = pd.Series(0.0, index=net_annual.index)

    formal_mask = is_formal == 1
    informal_mask = ~formal_mask

    # Formales
    gross[formal_mask] = gross_up_labor(net_annual[formal_mask], brackets)
    isr[formal_mask] = gross[formal_mask] - net_annual[formal_mask]  # ISR = bruto - neto

    # Informales
    gross[informal_mask] = net_annual[informal_mask]
    # isr queda en 0

    return gross, isr

# Bloque 3 – Definición de estructuras y tipos (para futuro)

In [84]:
# =====================================================
# BLOQUE 2
# DEFINIR ESTRUCTURA Y TIPOS
# =====================================================

# -----------------------------
# IDs
# -----------------------------
ID_TYPES = {
    "folioviv": "string",
    "foliohog": "string",
    "numren": "int64"
}

# -----------------------------
# POBLACION
# -----------------------------
POBLACION_COLUMNS = [
    "folioviv",
    "foliohog",
    "numren",
    "sexo",
    "edad",
    "parentesco"
]

POBLACION_TYPES = {
    "sexo": "int64",
    "edad": "int64",
    "parentesco": "int64"
}

# -----------------------------
# CONCENTRADO HOGAR
# -----------------------------
HOGAR_COLUMNS = [
    "folioviv",
    "foliohog",
    "ubica_geo",
    "est_dis",
    "upm",
    "factor"
]

HOGAR_TYPES = {
    "ubica_geo": "int64",
    "est_dis": "int64",
    "upm": "int64",
    "factor": "float64"
}

# -----------------------------
# INCOME (final columns)
# -----------------------------
INCOME_COLUMNS = [
    "folioviv", "foliohog", "numren", "year",
    "formal_trabajo_1", "formal_trabajo_2",
    "mi_trabajo_1", "mi_trabajo_2",
    "mi_capital", "mi_private_transfers",
    "market_income", "net_market_income",
    "di_direct_transfers", "disposable_income"
]

INCOME_TYPES = {
    "formal_trabajo_1": "int64",
    "formal_trabajo_2": "int64",
    "mi_trabajo_1": "float64",
    "mi_trabajo_2": "float64",
    "mi_capital": "float64",
    "mi_private_transfers": "float64",
    "market_income": "float64",
    "net_market_income": "float64",
    "di_direct_transfers": "float64",
    "disposable_income": "float64"
}
print("Structure defined successfully")

Structure defined successfully


# Bloque 4 - Tabla de formalidad

In [ ]:
# =====================================================
# BLOQUE 4A – CONSTRUIR FORMALIDAD LABORAL POR AÑO
# =====================================================
import pandas as pd

for year in [2018, 2024]:
    print(f"\n{'='*30}")
    print(f"Formalidad laboral {year}")
    print(f"{'='*30}")

    # Cargar tabla de trabajo
    trab = pd.read_csv(
        RAW_DIR / f"trabajos_{year}.csv",
        dtype={"folioviv": "string", "foliohog": "string", "numren": "int64"},
        low_memory=False
    )

    # Solo trabajos principal (id_trabajo==1) y secundario (id_trabajo==2)
    trab = trab[trab["id_trabajo"].isin([1, 2])]

    # Identificar subordinado 
    subordinado = trab["subor"] == 1

    # Seguridad social: IMSS, ISSSTE, Pemex, Ejército 
    seguro_social = (
        trab["medtrab_1"].eq(1) |
        trab["medtrab_2"].eq(1) |
        trab["medtrab_3"].eq(1) |
        trab["medtrab_4"].eq(1)
    )

    # Fondo de retiro (cs_p03_1)
    fondo_retiro = trab["pres_8"] == 8

    # Formal = subordinado Y (seguro social O fondo de retiro)
    trab["formal"] = (subordinado & (seguro_social | fondo_retiro)).astype(int)

    # Pivotear para obtener formal_trabajo_1 y formal_trabajo_2 por persona
    formal_pivot = trab.pivot_table(
        index=["folioviv", "foliohog", "numren"],
        columns="id_trabajo",
        values="formal",
        aggfunc="max"  # en caso de duplicados (no debería), tomamos 1 si al menos uno es formal
    ).reset_index()

    formal_pivot.columns = ["folioviv", "foliohog", "numren", "formal_trabajo_1", "formal_trabajo_2"]
    formal_pivot["formal_trabajo_1"] = formal_pivot["formal_trabajo_1"].fillna(0).astype(int)
    formal_pivot["formal_trabajo_2"] = formal_pivot["formal_trabajo_2"].fillna(0).astype(int)

    # Guardar temporal
    out = PROCESSED_DIR / f"formalidad_{year}.csv"
    formal_pivot.to_csv(out, index=False)
    print(f"Guardado {out.name} - filas: {len(formal_pivot)}")


Formalidad laboral 2018
Guardado formalidad_2018.csv - filas: 127638

Formalidad laboral 2024
Guardado formalidad_2024.csv - filas: 150382


# Bloque 5 – Procesamiento ENIGH 2018

In [86]:
# =====================================================
# BLOQUE 5 – INGRESOS 2018
# =====================================================
year = 2018
print(f"\n{'='*30}")
print(f"Processing {year}")
print(f"{'='*30}")

# ---- Cargar ingresos ----
df = pd.read_csv(
    RAW_DIR / f"ingresos_{year}.csv",
    dtype={
        "folioviv": "string",
        "foliohog": "string",
        "numren": "int64",
        "clave": "string",
        "ing_tri": "float64"
    },
    low_memory=False
)
print(f"Rows raw: {len(df)}")

# ---- Cargar formalidad  ----
form = pd.read_csv(
    PROCESSED_DIR / f"formalidad_{year}.csv",
    dtype={
        "folioviv": "string",
        "foliohog": "string",
        "numren": "int64"
    }
)

# ---- Clasificar registros de ingreso ----
mask_main = df["clave"].isin(LABOR_KEYS_MAIN)
mask_sec  = df["clave"].isin(LABOR_KEYS_SECONDARY)
mask_other = df["clave"].isin(LABOR_KEYS_OTHER)
mask_capital = df["clave"].isin(CAPITAL_KEYS)
mask_private = df["clave"].isin(PRIVATE_TRANSFER_KEYS)
mask_direct = df["clave"].isin(DIRECT_TRANSFERS_2018)

# ---- Sumar ingreso trimestral por persona para cada categoría ----
net_main_tri = (
    df.loc[mask_main]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("net_trabajo1_tri")
).reset_index()

net_sec_tri = (
    df.loc[mask_sec]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("net_trabajo2_tri")
).reset_index()

net_other_tri = (
    df.loc[mask_other]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("net_otros_laborales_tri")
).reset_index()

mi_capital_tri = (
    df.loc[mask_capital]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("mi_capital_tri")
).reset_index()

mi_private_tri = (
    df.loc[mask_private]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("mi_private_transfers_tri")
).reset_index()

di_direct_tri = (
    df.loc[mask_direct]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("di_direct_transfers_tri")
).reset_index()

# ---- Construir base persona ----
person = df[["folioviv", "foliohog", "numren"]].drop_duplicates()
for tbl in [net_main_tri, net_sec_tri, net_other_tri, mi_capital_tri, mi_private_tri, di_direct_tri, form]:
    person = person.merge(tbl, on=["folioviv", "foliohog", "numren"], how="left")

# Rellenar NaN con 0
for col in person.columns:
    if person[col].dtype.kind in 'fc':
        person[col] = person[col].fillna(0)

# ---- Anualizar (multiplicar trimestral por 4) ----
cols_tri = [
    "net_trabajo1_tri", "net_trabajo2_tri", "net_otros_laborales_tri",
    "mi_capital_tri", "mi_private_transfers_tri", "di_direct_transfers_tri"
]
for c in cols_tri:
    person[c.replace("_tri", "_ann")] = person[c] * 4

# Renombrar columnas anuales para claridad
person = person.rename(columns={
    "net_trabajo1_tri_ann": "net_trabajo1_ann",
    "net_trabajo2_tri_ann": "net_trabajo2_ann",
    "net_otros_laborales_tri_ann": "net_otros_laborales_ann",
    "mi_capital_tri_ann": "mi_capital_ann",
    "mi_private_transfers_tri_ann": "mi_private_transfers_ann",
    "di_direct_transfers_tri_ann": "di_direct_transfers_ann"
})

# ---- Calcular bruto e ISR por trabajo según formalidad ----
brackets = TAX_BRACKETS[year]

gross1, isr1 = compute_gross_and_tax(
    person["net_trabajo1_ann"], person["formal_trabajo_1"], brackets
)
gross2, isr2 = compute_gross_and_tax(
    person["net_trabajo2_ann"], person["formal_trabajo_2"], brackets
)

# Otros laborales: siempre informales -> bruto = neto, ISR = 0
gross_other = person["net_otros_laborales_ann"]
isr_other = 0.0

person["mi_trabajo_1"] = gross1
person["mi_trabajo_2"] = gross2
person["mi_trabajo_total"] = gross1 + gross2 + gross_other
person["isr_total"] = isr1 + isr2

# ---- Ingresos de mercado y neto ----
person["market_income"] = (
    person["mi_trabajo_total"] +
    person["mi_capital_ann"] +
    person["mi_private_transfers_ann"]
)
person["net_market_income"] = person["market_income"] - person["isr_total"]
person["disposable_income"] = person["net_market_income"] + person["di_direct_transfers_ann"]

# ---- Deflactar (todas las monetarias ya son anuales) ----
monetary_cols = [
    "mi_trabajo_1", "mi_trabajo_2", "mi_trabajo_total",
    "mi_capital_ann", "mi_private_transfers_ann",
    "market_income", "net_market_income",
    "di_direct_transfers_ann", "disposable_income"
]
person[monetary_cols] *= DEFLATORS[year]

# ---- Columnas finales ----
person["year"] = year
person = person.rename(columns={
    "mi_capital_ann": "mi_capital",
    "mi_private_transfers_ann": "mi_private_transfers",
    "di_direct_transfers_ann": "di_direct_transfers"
})

df_2018_out = person[[
    "folioviv", "foliohog", "numren", "year",  "net_trabajo1_ann", "net_trabajo2_ann", "isr_total", 
    "formal_trabajo_1", "formal_trabajo_2",
    "mi_trabajo_1", "mi_trabajo_2",
    "mi_capital", "mi_private_transfers",
    "market_income", "net_market_income",
    "di_direct_transfers", "disposable_income"
]]

# ---- Guardar ----
output = PROCESSED_DIR / f"income_components_{year}.csv"
df_2018_out.to_csv(output, index=False)
print(f"Filas: {len(df_2018_out)} - Guardado {output.name}")


Processing 2018
Rows raw: 348487
Filas: 182434 - Guardado income_components_2018.csv


# Bloque 6 – Procesamiento ENIGH 2024

In [87]:
# =====================================================
# BLOQUE 6 – INGRESOS 2024
# =====================================================
year = 2024
print(f"\n{'='*30}")
print(f"Processing {year}")
print(f"{'='*30}")

# ---- Cargar ingresos ----
df = pd.read_csv(
    RAW_DIR / f"ingresos_{year}.csv",
    dtype={
        "folioviv": "string",
        "foliohog": "string",
        "numren": "int64",
        "clave": "string",
        "ing_tri": "float64"
    },
    low_memory=False
)
print(f"Rows raw: {len(df)}")

# ---- Cargar formalidad  ----
form = pd.read_csv(
    PROCESSED_DIR / f"formalidad_{year}.csv",
    dtype={
        "folioviv": "string",
        "foliohog": "string",
        "numren": "int64"
    }
)

# ---- Clasificar registros de ingreso ----
mask_main = df["clave"].isin(LABOR_KEYS_MAIN)
mask_sec  = df["clave"].isin(LABOR_KEYS_SECONDARY)
mask_other = df["clave"].isin(LABOR_KEYS_OTHER)
mask_capital = df["clave"].isin(CAPITAL_KEYS)
mask_private = df["clave"].isin(PRIVATE_TRANSFER_KEYS)
mask_direct = df["clave"].isin(DIRECT_TRANSFERS_2024)

# ---- Sumar ingreso trimestral por persona para cada categoría ----
net_main_tri = (
    df.loc[mask_main]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("net_trabajo1_tri")
).reset_index()

net_sec_tri = (
    df.loc[mask_sec]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("net_trabajo2_tri")
).reset_index()

net_other_tri = (
    df.loc[mask_other]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("net_otros_laborales_tri")
).reset_index()

mi_capital_tri = (
    df.loc[mask_capital]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("mi_capital_tri")
).reset_index()

mi_private_tri = (
    df.loc[mask_private]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("mi_private_transfers_tri")
).reset_index()

di_direct_tri = (
    df.loc[mask_direct]
    .groupby(["folioviv", "foliohog", "numren"])["ing_tri"]
    .sum()
    .rename("di_direct_transfers_tri")
).reset_index()

# ---- Construir base persona ----
person = df[["folioviv", "foliohog", "numren"]].drop_duplicates()
for tbl in [net_main_tri, net_sec_tri, net_other_tri, mi_capital_tri, mi_private_tri, di_direct_tri, form]:
    person = person.merge(tbl, on=["folioviv", "foliohog", "numren"], how="left")

# Rellenar NaN con 0
for col in person.columns:
    if person[col].dtype.kind in 'fc':
        person[col] = person[col].fillna(0)

# ---- Anualizar (multiplicar trimestral por 4) ----
cols_tri = [
    "net_trabajo1_tri", "net_trabajo2_tri", "net_otros_laborales_tri",
    "mi_capital_tri", "mi_private_transfers_tri", "di_direct_transfers_tri"
]
for c in cols_tri:
    person[c.replace("_tri", "_ann")] = person[c] * 4

# Renombrar columnas anuales para claridad
person = person.rename(columns={
    "net_trabajo1_tri_ann": "net_trabajo1_ann",
    "net_trabajo2_tri_ann": "net_trabajo2_ann",
    "net_otros_laborales_tri_ann": "net_otros_laborales_ann",
    "mi_capital_tri_ann": "mi_capital_ann",
    "mi_private_transfers_tri_ann": "mi_private_transfers_ann",
    "di_direct_transfers_tri_ann": "di_direct_transfers_ann"
})

# ---- Calcular bruto e ISR por trabajo según formalidad ----
brackets = TAX_BRACKETS[year]

gross1, isr1 = compute_gross_and_tax(
    person["net_trabajo1_ann"], person["formal_trabajo_1"], brackets
)
gross2, isr2 = compute_gross_and_tax(
    person["net_trabajo2_ann"], person["formal_trabajo_2"], brackets
)

# Otros laborales: siempre informales -> bruto = neto, ISR = 0
gross_other = person["net_otros_laborales_ann"]
isr_other = 0.0

person["mi_trabajo_1"] = gross1
person["mi_trabajo_2"] = gross2
person["mi_trabajo_total"] = gross1 + gross2 + gross_other
person["isr_total"] = isr1 + isr2

# ---- Ingresos de mercado y neto ----
person["market_income"] = (
    person["mi_trabajo_total"] +
    person["mi_capital_ann"] +
    person["mi_private_transfers_ann"]
)
person["net_market_income"] = person["market_income"] - person["isr_total"]
person["disposable_income"] = person["net_market_income"] + person["di_direct_transfers_ann"]

# ---- Deflactar (todas las monetarias ya son anuales) ----
monetary_cols = [
    "mi_trabajo_1", "mi_trabajo_2", "mi_trabajo_total",
    "mi_capital_ann", "mi_private_transfers_ann",
    "market_income", "net_market_income",
    "di_direct_transfers_ann", "disposable_income"
]
person[monetary_cols] *= DEFLATORS[year]

# ---- Columnas finales ----
person["year"] = year
person = person.rename(columns={
    "mi_capital_ann": "mi_capital",
    "mi_private_transfers_ann": "mi_private_transfers",
    "di_direct_transfers_ann": "di_direct_transfers"
})

df_2024_out = person[[
    "folioviv", "foliohog", "numren", "year", "net_trabajo1_ann", "net_trabajo2_ann", "isr_total", 
    "formal_trabajo_1", "formal_trabajo_2",
    "mi_trabajo_1", "mi_trabajo_2",
    "mi_capital", "mi_private_transfers",
    "market_income", "net_market_income",
    "di_direct_transfers", "disposable_income"
]]

# ---- Guardar ----
output = PROCESSED_DIR / f"income_components_{year}.csv"
df_2024_out.to_csv(output, index=False)
print(f"Filas: {len(df_2024_out)} - Guardado {output.name}")


Processing 2024
Rows raw: 391563
Filas: 202365 - Guardado income_components_2024.csv


# Bloque 7 – Concatenación 2018 + 2024 y dataset final

In [88]:
# =====================================================
# BLOQUE 7 – CONCATENACIÓN 2018 + 2024
# =====================================================
df_2018 = pd.read_csv(PROCESSED_DIR / "income_components_2018.csv")
df_2024 = pd.read_csv(PROCESSED_DIR / "income_components_2024.csv")
df_final = pd.concat([df_2018, df_2024], axis=0, ignore_index=True)

print("\nFinal dataset shape:", df_final.shape)
print("Year distribution:\n", df_final["year"].value_counts().sort_index())

# Diagnósticos
print("\nMissing values per column:")
print(df_final.isna().sum())
print("\nIncome summary by year:")
print(df_final.groupby("year")[["market_income","net_market_income","disposable_income"]].describe())

df_final.to_csv(PROCESSED_DIR / "income_fiscal_incidence_base.csv", index=False)
print("Saved income_fiscal_incidence_base.csv")


Final dataset shape: (384799, 17)
Year distribution:
 year
2018    182434
2024    202365
Name: count, dtype: int64

Missing values per column:
folioviv                0
foliohog                0
numren                  0
year                    0
net_trabajo1_ann        0
net_trabajo2_ann        0
isr_total               0
formal_trabajo_1        0
formal_trabajo_2        0
mi_trabajo_1            0
mi_trabajo_2            0
mi_capital              0
mi_private_transfers    0
market_income           0
net_market_income       0
di_direct_transfers     0
disposable_income       0
dtype: int64

Income summary by year:
     market_income                                                 \
             count          mean            std  min          25%   
year                                                                
2018      182434.0  55897.638920  130152.020616  0.0  1967.200000   
2024      202365.0  75043.896405  175784.147731  0.0  6091.780647   

                              